In [1]:
import sys
print(sys.executable)

/Users/macbook/MyProjects/ai_proctor/venv/bin/python


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import onnxruntime as ort
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from tqdm import tqdm

In [3]:
BASE_DIR = os.path.dirname(os.getcwd()) # notebook in app/notebooks/
MODELS_DIR = os.path.join(BASE_DIR, "models")
DATASET_DIR = os.path.join(BASE_DIR, "dataset")


MINIFASNET_PATH = os.path.join(MODELS_DIR, "MiniFASNetV2.onnx")
FACEBAGNET_PATH = os.path.join(MODELS_DIR, "facebagnet_color_96.onnx")


# MiniFASNet expects 80x80, FaceBagNet expects 96x96
IMAGE_SIZE_MINI = (80, 80)
IMAGE_SIZE_FACEBAG = (96, 96)
BATCH_SIZE = 32


print(f"Checking Models...\nMiniFASNet: {os.path.exists(MINIFASNET_PATH)}\nFaceBagNet: {os.path.exists(FACEBAGNET_PATH)}")

Checking Models...
MiniFASNet: True
FaceBagNet: True


In [9]:
def load_dataset(dataset_dir, image_size):
    X, y = [], []
    # Alphabetical order: 'fake' will be 0, 'real' will be 1
    for label in ["fake", "real"]:
        p = os.path.join(dataset_dir, label)
        if not os.path.isdir(p): 
            print(f"Warning: Directory {p} not found.")
            continue
        for f in os.listdir(p):
            img_path = os.path.join(p, f)
            img = cv2.imread(img_path)
            if img is None: continue
            
            # Models expect RGB, OpenCV loads BGR
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, image_size)
            X.append(img)
            y.append(label)
    return np.array(X), np.array(y)

print("Loading datasets...")
X_mini_raw, y = load_dataset(DATASET_DIR, IMAGE_SIZE_MINI)
X_face_raw, _ = load_dataset(DATASET_DIR, IMAGE_SIZE_FACEBAG)

le = LabelEncoder()
y_true = le.fit_transform(y) # fake=0, real=1

print(f"Total Samples: {len(X_mini_raw)}")
print(f"Classes: {le.classes_} -> mapped to [0, 1]")


Loading datasets...
Total Samples: 269
Classes: ['fake' 'real'] -> mapped to [0, 1]


In [10]:
def preprocess_mini(X):
    # MiniFASNetV2 usually uses 127.5 scale in 2026 production
    X = (X.astype(np.float32) - 127.5) / 128.0
    return np.transpose(X, (0, 3, 1, 2))

def preprocess_facebag(X):
    X = X.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    X = (X - mean) / std
    return np.transpose(X, (0, 3, 1, 2))

X_mini = preprocess_mini(X_mini_raw)
X_face = preprocess_facebag(X_face_raw)

session = ort.InferenceSession(MINIFASNET_PATH)
print(session.get_outputs()[0].shape)

['batch_size', 3]


In [5]:
pip install tdqm onnxruntime scikit-learn pandas


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import numpy as np
import onnxruntime as ort
from tqdm import tqdm

def run_inference(model_path, X_processed, model_name="MiniFASNet",
                  threshold_live=0.5):
    session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name

    y_pred = []
    y_score = []   # ⭐ NEW: continuous live score

    print(f"Running inference for {model_name}...")
    for i in tqdm(range(len(X_processed))):
        inp = np.expand_dims(X_processed[i], axis=0)
        logits = session.run(None, {input_name: inp})[0][0]

        if model_name == "FaceBagNet":
            # logits: [fake, real] (based on your backend logic)
            score_diff = logits[0] - logits[1]
            live_score = 1.0 / (1.0 + np.exp(score_diff / 250.0))
            prediction = 1 if live_score > 0.90 else 0

        else:  # MiniFASNet
            fake_logit = logits[0]
            real_logit = logits[1]

            exps = np.exp([fake_logit, real_logit])
            probs = exps / np.sum(exps)

            live_score = probs[1]              # ⭐ probability of LIVE
            prediction = 1 if live_score > threshold_live else 0

        y_score.append(live_score)
        y_pred.append(prediction)

    return np.array(y_pred), np.array(y_score)


In [ ]:
def compute_pad_metrics(y_true, y_score, threshold):
    """
    y_true: 1 = live, 0 = spoof
    y_score: probability of live
    """

    y_pred = (y_score >= threshold).astype(int)

    spoof_mask = (y_true == 0)
    live_mask = (y_true == 1)

    apcer = np.sum((y_pred == 1) & spoof_mask) / np.sum(spoof_mask)
    bpcer = np.sum((y_pred == 0) & live_mask) / np.sum(live_mask)
    acer = (apcer + bpcer) / 2

    return apcer, bpcer, acer


from sklearn.metrics import roc_curve, auc

def compute_roc_eer(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2

    return {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "auc": roc_auc,
        "eer": eer,
        "eer_threshold": thresholds[idx]
    }


In [ ]:
# Inference for FaceBagNet
y_pred, y_score = run_inference(
    FACEBAGNET_PATH,
    X_face,
    model_name="FaceBagNet",
    threshold_live=0.5
)


# ROC + EER
roc_res = compute_roc_eer(y_true, y_score)
print(f"AUC: {roc_res['auc']:.4f}")
print(f"EER: {roc_res['eer']:.4f}")

# PAD metrics at EER threshold
apcer, bpcer, acer = compute_pad_metrics(
    y_true,
    y_score,
    roc_res["eer_threshold"]
)

print(f"APCER: {apcer:.4f}")
print(f"BPCER: {bpcer:.4f}")
print(f"ACER : {acer:.4f}")


Running inference for FaceBagNet...


100%|██████████| 269/269 [00:02<00:00, 100.33it/s]

AUC: 0.9760
EER: 0.0662
APCER: 0.0686
BPCER: 0.0638
ACER : 0.0662


In [14]:
# Inference for MiniFasNet
y_pred, y_score = run_inference(
    MINIFASNET_PATH,
    X_mini,
    model_name="MiniFASNet",
    threshold_live=0.5
)


# ROC + EER
roc_res = compute_roc_eer(y_true, y_score)
print(f"AUC: {roc_res['auc']:.4f}")
print(f"EER: {roc_res['eer']:.4f}")

# PAD metrics at EER threshold
apcer, bpcer, acer = compute_pad_metrics(
    y_true,
    y_score,
    roc_res["eer_threshold"]
)

print(f"APCER: {apcer:.4f}")
print(f"BPCER: {bpcer:.4f}")
print(f"ACER : {acer:.4f}")


Running inference for MiniFASNet...


100%|██████████| 269/269 [00:00<00:00, 598.20it/s]

AUC: 0.4179
EER: 0.5096
APCER: 0.5086
BPCER: 0.5106
ACER : 0.5096
